### 1. Preprocess a text:
a. Tokenize the text (file/query) to obtain a list of tokens. You can use an already existing tokenizer library.

In [3]:
import os.path

import nltk
# nltk.download('punkt')
from os import listdir
from os.path import isfile, join

mypath = 'dataset/'


def tokenize(text):
    tokens = nltk.word_tokenize(text)
    #print(tokens)
    return tokens


onlyfiles = [f for f in listdir(mypath) if isfile(join(mypath, f))]
texts = []

# cette variable files_order sert à faire l'indexing
files_order = []
for x in onlyfiles:
    if x.endswith('txt'):
        files_order.append(os.path.splitext(os.path.basename(x))[0])
        with open(mypath + x, 'r') as file:
            data = file.read().replace('\n', '')
            # print(data)
            texts.append(tokenize(data))

# print(files_order[0], files_order[10])
# print('\n')
print("Contenu  fichiers : ", texts)


Contenu  fichiers :  [['A', 'man', 'with', 'a', 'faded', ',', 'well-worn', 'notebook', 'open', 'in', 'his', 'lap', '.', 'A', 'woman', 'experiencing', 'a', 'morning', 'ritual', 'she', 'doesnâ€™t', 'understand', '.', 'Until', 'he', 'begins', 'to', 'read', 'to', 'her.The', 'Notebook', 'is', 'an', 'achingly', 'tender', 'story', 'about', 'the', 'enduring', 'power', 'of', 'love', ',', 'a', 'story', 'of', 'miracles', 'that', 'will', 'stay', 'with', 'you', 'forever', '.', 'Set', 'amid', 'the', 'austere', 'beauty', 'of', 'coastal', 'North', 'Carolina', 'in', '1946', ',', 'The', 'Notebook', 'begins', 'with', 'the', 'story', 'of', 'Noah', 'Calhoun', ',', 'a', 'poor', ',', 'rural', 'Southerner', 'returned', 'home', 'from', 'World', 'War', 'II', '.', 'Noah', ',', 'thirty-one', ',', 'is', 'restoring', 'a', 'plantation', 'home', 'to', 'its', 'former', 'glory', ',', 'and', 'he', 'is', 'haunted', 'by', 'images', 'of', 'the', 'beautiful', 'girl', 'he', 'met', 'fourteen', 'years', 'earlier', ',', 'a', 'r

b. Perform stemming/lemmatization, stopwords removal (find a list 
of stopwords online), as well as turning all tokens to lower case 
(you decide for the ordering of these steps). 

In [4]:
def lower_tokens(texts):
    texts_lower = []
    for text in texts:
        phrase = []
        for word in text:
            phrase.append(word.lower())
        texts_lower.append(phrase)
    return texts_lower


# nltk.download('stopwords')
from nltk.corpus import stopwords

stops = stopwords.words('english')

#print(stops)

def remove_stopwords(texts):
    for text in texts:
        for word in text:
            if word in stops:
                text.remove(word)
    return texts


texts_lower = lower_tokens(texts)
print("Contenu minuscule : ", texts_lower)
print('\n')

texts_stopword = remove_stopwords(texts_lower)
print("Contenu stopwords : ", texts_stopword)
print('\n')

from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize

ps = PorterStemmer()

def stemming(texts):
    texts_stem = []
    for text in texts:
        phrase = []
        for word in text:
            phrase.append(ps.stem(word))
        texts_stem.append(phrase)
    return texts_stem


texts_preprocessed = stemming(texts_stopword)
print("Contenu  stemming : ", texts_preprocessed)

Contenu minuscule :  [['a', 'man', 'with', 'a', 'faded', ',', 'well-worn', 'notebook', 'open', 'in', 'his', 'lap', '.', 'a', 'woman', 'experiencing', 'a', 'morning', 'ritual', 'she', 'doesnâ€™t', 'understand', '.', 'until', 'he', 'begins', 'to', 'read', 'to', 'her.the', 'notebook', 'is', 'an', 'achingly', 'tender', 'story', 'about', 'the', 'enduring', 'power', 'of', 'love', ',', 'a', 'story', 'of', 'miracles', 'that', 'will', 'stay', 'with', 'you', 'forever', '.', 'set', 'amid', 'the', 'austere', 'beauty', 'of', 'coastal', 'north', 'carolina', 'in', '1946', ',', 'the', 'notebook', 'begins', 'with', 'the', 'story', 'of', 'noah', 'calhoun', ',', 'a', 'poor', ',', 'rural', 'southerner', 'returned', 'home', 'from', 'world', 'war', 'ii', '.', 'noah', ',', 'thirty-one', ',', 'is', 'restoring', 'a', 'plantation', 'home', 'to', 'its', 'former', 'glory', ',', 'and', 'he', 'is', 'haunted', 'by', 'images', 'of', 'the', 'beautiful', 'girl', 'he', 'met', 'fourteen', 'years', 'earlier', ',', 'a', 'r

### 2. Create inverted index for your files. You do not need to add the positions of the occurrences of the terms.

In [5]:
def words(texts):
    all_words = []
    for text in texts:
        for word in text:
            all_words.append(word)
    # print(len(all_words))
    # on convertit en ensemble pour avoir chaque mot une seule fois
    all_words = set(all_words)
    # print(len(all_words))
    return all_words

#liste de mot unique
words = words(texts_preprocessed)


def index(texts, words):
    index = []
    # on parcourt chaque mot unique
    for word in words:
        i = 0
        compteurDeDocument = 0
        dicos = []
        # on parcourt le texte de chaque fichier qu'on a preprocess
        for text in texts:
            num_file = files_order[i] # contient l'ordre dans lequel les fichiers sont lus
            compteurDeMotDansFicher = 0
            # on parcourt chaque mot de fichier
            for word2 in text:
                # on compare le mot lu avec le second mot lu
                if word == word2:
                    compteurDeMotDansFicher += 1
                    # on ajoute +1 au compteur de document 1 seule fois par fichier donc au 1er mot trouvé
                    if compteurDeMotDansFicher == 1: 
                        compteurDeDocument += 1
            # on crée un dictionnaire pour le fichier contenant le mot lu
            if compteurDeMotDansFicher > 0:
                dico = dict(num_doc=num_file, nbMots=compteurDeMotDansFicher)
                dicos.append(dico)
            i += 1
        dicoDeListeDeDico = dict(word=word, nbDocument=compteurDeDocument, dicos=dicos)
        index.append(dicoDeListeDeDico)
    return index


indexing = index(texts_preprocessed, words)
for i in indexing:
    print(i)


{'word': 'minist', 'nbDocument': 1, 'dicos': [{'num_doc': '4', 'nbMots': 1}]}
{'word': 'viscer', 'nbDocument': 1, 'dicos': [{'num_doc': '10', 'nbMots': 1}]}
{'word': 'alli', 'nbDocument': 1, 'dicos': [{'num_doc': '1', 'nbMots': 3}]}
{'word': 'street', 'nbDocument': 1, 'dicos': [{'num_doc': '11', 'nbMots': 2}]}
{'word': 'later', 'nbDocument': 1, 'dicos': [{'num_doc': '7', 'nbMots': 1}]}
{'word': 'scatter', 'nbDocument': 1, 'dicos': [{'num_doc': '9', 'nbMots': 1}]}
{'word': 'remain', 'nbDocument': 1, 'dicos': [{'num_doc': '1', 'nbMots': 1}]}
{'word': 'true', 'nbDocument': 2, 'dicos': [{'num_doc': '10', 'nbMots': 1}, {'num_doc': '11', 'nbMots': 1}]}
{'word': 'cultur', 'nbDocument': 1, 'dicos': [{'num_doc': '9', 'nbMots': 1}]}
{'word': 'quest', 'nbDocument': 1, 'dicos': [{'num_doc': '10', 'nbMots': 1}]}
{'word': 'oakmont', 'nbDocument': 1, 'dicos': [{'num_doc': '11', 'nbMots': 1}]}
{'word': 'action', 'nbDocument': 1, 'dicos': [{'num_doc': '7', 'nbMots': 1}]}
{'word': 'spark', 'nbDocument':

### 3. Compute the term frequency tf_t,d of each term in each of the files. Compute the inverted document frequency idf_t of each term. 

In [6]:
import pandas as pd

def tf(unique_words, all_documents):
    # Step 3 and 4: Create term frequency matrix with 0s and 1s
    term_frequency_list = []
    for word in unique_words:
        word_row = []
        for doc in all_documents:
            count = 0
            # Check if the word appears in the document
            word_found = False
            for result in indexing:
                for doc_item in result['dicos']:
                    if doc == doc_item['num_doc'] and word == result['word']:
                        word_row.append(doc_item['nbMots'])
                        word_found = True
                        break
                if word_found:
                    break
            if not word_found:
                word_row.append(0)
        term_frequency_list.append(word_row)
    return term_frequency_list


# Step 1: Extract unique words from the preprocessed texts (already done)
unique_words_list = list(words)

# Step 2: Get all documents from files_order
all_documents = files_order

term_frequency_list = tf(unique_words_list, all_documents)

# Créer le DataFrame à partir de la matrice de fréquence de termes et des noms de colonnes et de lignes
tf_matrix = pd.DataFrame(term_frequency_list, columns=all_documents, index=unique_words_list)

# Afficher le DataFrame
print("TF Matrix:")
print(tf_matrix)


TF Matrix:
         1  10  11  2  3  4  5  6  7  8  9
minist   0   0   0  0  0  1  0  0  0  0  0
viscer   0   1   0  0  0  0  0  0  0  0  0
alli     3   0   0  0  0  0  0  0  0  0  0
street   0   0   2  0  0  0  0  0  0  0  0
later    0   0   0  0  0  0  0  0  1  0  0
...     ..  ..  .. .. .. .. .. .. .. .. ..
pellet   0   0   0  0  0  0  0  1  0  0  0
na'vi    0   0   0  0  0  0  0  0  0  0  2
attitud  0   0   0  0  0  0  0  1  0  0  0
of       0   0   0  0  0  0  0  0  1  0  0
town     1   0   0  0  0  0  0  0  0  0  0

[635 rows x 11 columns]


In [7]:
import math

def calculate_idf(indexing, total_documents):
    idf_dict = {}
    for result in indexing:
        word = result['word']
        num_documents_with_word = result['nbDocument']
        idf = math.log(total_documents / num_documents_with_word)
        idf_dict[word] = idf
    return idf_dict

# Step 1: Calculate the total number of documents in your corpus
total_documents = len(all_documents)

# Step 2: Calculate IDF for each word
idf_dict = calculate_idf(indexing, total_documents)

# Print IDF for each word
print("IDF :")
for word, idf in idf_dict.items():
    print(f"Word: {word}, IDF: {idf}")


IDF :
Word: minist, IDF: 2.3978952727983707
Word: viscer, IDF: 2.3978952727983707
Word: alli, IDF: 2.3978952727983707
Word: street, IDF: 2.3978952727983707
Word: later, IDF: 2.3978952727983707
Word: scatter, IDF: 2.3978952727983707
Word: remain, IDF: 2.3978952727983707
Word: true, IDF: 1.7047480922384253
Word: cultur, IDF: 2.3978952727983707
Word: quest, IDF: 2.3978952727983707
Word: oakmont, IDF: 2.3978952727983707
Word: action, IDF: 2.3978952727983707
Word: spark, IDF: 2.3978952727983707
Word: sulli, IDF: 2.3978952727983707
Word: against, IDF: 2.3978952727983707
Word: unit, IDF: 2.3978952727983707
Word: year, IDF: 1.2992829841302609
Word: colonel, IDF: 2.3978952727983707
Word: can, IDF: 2.3978952727983707
Word: act, IDF: 2.3978952727983707
Word: extraordinari, IDF: 2.3978952727983707
Word: stop, IDF: 2.3978952727983707
Word: mountain, IDF: 2.3978952727983707
Word: plantat, IDF: 2.3978952727983707
Word: power, IDF: 1.7047480922384253
Word: imag, IDF: 2.3978952727983707
Word: lovejoy, 

### 4. Implement the retrieval models: tf-idf vector model, BM25. The output of each model should be a ranked list of documents (in descending score order).

In [8]:
#tf-idf vector model
import pandas as pd

def tf_idf(tf_matrix, idf_dict):
    tf_idf_matrix = []
    for row_label, row in tf_matrix.iterrows():
        tf_idf_row = []
        for word, tf_value in row.items():
            tf_idf_row.append(tf_value * idf_dict[row_label])
        tf_idf_matrix.append(tf_idf_row)
    return pd.DataFrame(tf_idf_matrix, columns=tf_matrix.columns, index=tf_matrix.index)

# Step 4: Calculate TF-IDF for each document
tf_idf_matrix = tf_idf(tf_matrix, idf_dict)
tf_idf_matrix.to_csv('tf_idf.csv')

# Print TF-IDF Matrix
print("TF-IDF Matrix:")
print(tf_idf_matrix)


TF-IDF Matrix:
                1        10        11    2    3         4    5         6  \
minist   0.000000  0.000000  0.000000  0.0  0.0  2.397895  0.0  0.000000   
viscer   0.000000  2.397895  0.000000  0.0  0.0  0.000000  0.0  0.000000   
alli     7.193686  0.000000  0.000000  0.0  0.0  0.000000  0.0  0.000000   
street   0.000000  0.000000  4.795791  0.0  0.0  0.000000  0.0  0.000000   
later    0.000000  0.000000  0.000000  0.0  0.0  0.000000  0.0  0.000000   
...           ...       ...       ...  ...  ...       ...  ...       ...   
pellet   0.000000  0.000000  0.000000  0.0  0.0  0.000000  0.0  2.397895   
na'vi    0.000000  0.000000  0.000000  0.0  0.0  0.000000  0.0  0.000000   
attitud  0.000000  0.000000  0.000000  0.0  0.0  0.000000  0.0  2.397895   
of       0.000000  0.000000  0.000000  0.0  0.0  0.000000  0.0  0.000000   
town     2.397895  0.000000  0.000000  0.0  0.0  0.000000  0.0  0.000000   

                7    8         9  
minist   0.000000  0.0  0.000000  
vi

In [56]:
#ranked list for tf-idf
requete = "Romantic movies about love between rich and poor."

query_token = []

query_token.append(tokenize(requete))
print(query_token)


query_lower = lower_tokens(query_token)
print(query_lower)

query_stopword = remove_stopwords(query_lower)
print(query_stopword)

query_stem = stemming(query_stopword)
print(query_stem)

query = query_stem[0]
print(query)

def tfidf_score(tf_idf_matrix, query):
     # Initialiser une liste pour stocker les scores TF-IDF pour chaque document
    tfidf_scores = []
    
    # Pour chaque document dans la matrice TF-IDF
    for document_index in tf_idf_matrix.columns:
        # Initialiser le score TF-IDF pour ce document à 0
        document_score = 0
        
        # Pour chaque mot-clé dans la requête
        for keyword in query:
            # Vérifier si le mot-clé est présent dans la matrice TF-IDF
            if keyword in tf_idf_matrix.index:
                # Ajouter le poids TF-IDF correspondant à ce mot-clé pour ce document
                document_score += tf_idf_matrix.loc[keyword, document_index]
        
        # Ajouter le score TF-IDF calculé pour ce document à la liste des scores
        tfidf_scores.append((document_index, document_score))
       
    # Trier les documents par score TF-IDF, en ordre décroissant
    ranked_documents_tfidf = sorted(tfidf_scores, key=lambda x: x[1], reverse=True)
    
    return ranked_documents_tfidf
    
    
# Calculer le score TF-IDF pour chaque document
ranked_documents_tfidf = tfidf_score(tf_idf_matrix, query)

print()
# Affichage des résultats
for doc_id, score in ranked_documents_tfidf:
    print(f"Document {doc_id}: TF-IDF Score = {score}")


[['Romantic', 'movies', 'about', 'love', 'between', 'rich', 'and', 'poor', '.']]
[['romantic', 'movies', 'about', 'love', 'between', 'rich', 'and', 'poor', '.']]
[['romantic', 'movies', 'love', 'rich', 'poor', '.']]
[['romant', 'movi', 'love', 'rich', 'poor', '.']]
['romant', 'movi', 'love', 'rich', 'poor', '.']

Document 1: TF-IDF Score = 3.9852929981564467
Document 8: TF-IDF Score = 1.9155711591645943
Document 2: TF-IDF Score = 1.6642567308836882
Document 11: TF-IDF Score = 1.6177367152487954
Document 9: TF-IDF Score = 1.463586035421537
Document 10: TF-IDF Score = 1.0581209273133727
Document 7: TF-IDF Score = 1.0581209273133727
Document 4: TF-IDF Score = 0.9039702474861144
Document 6: TF-IDF Score = 0.6061358035703155
Document 3: TF-IDF Score = 0.0
Document 5: TF-IDF Score = 0.0


In [10]:
#bm-25 model et ranked list

# Calculer les longueurs des documents
document_lengths = {}
for i, document in enumerate(texts_preprocessed, start=1):
    doc_id = files_order[i - 1]  # Récupérer l'identifiant du document à partir de files_order
    document_lengths[doc_id] = len(document)

# Calculer la longueur moyenne des documents
avg_document_length = sum(document_lengths.values()) / len(document_lengths)

# Définir les paramètres du modèle BM25
k1 = 1.5
b = 0.75

# Calculer le score BM25 pour chaque document par rapport à la requête
scores_bm25 = {}
for i, document in enumerate(texts_preprocessed, start=1):
    doc_id = files_order[i - 1]  # Récupérer l'identifiant du document à partir de files_order
    score = 0
    for word in query:
        # Calculer les composantes du score BM25 pour ce mot
        tf = document.count(word)
        idf = idf_dict.get(word, 0)  # Obtenir l'IDF du mot s'il existe, sinon 0
        doc_length = document_lengths[doc_id]
        score += (idf * tf * (k1 + 1)) / (tf + k1 * (1 - b + b * (doc_length / avg_document_length)))
    scores_bm25[doc_id] = score

# Trier les documents par score BM25 décroissant
ranked_documents_bm25 = sorted(scores_bm25.items(), key=lambda x: x[1], reverse=True)

# Afficher les documents classés selon le score BM25
for doc_id, score in ranked_documents_bm25:
    print(f"Document {doc_id}: BM25 Score = {score}")


Document 8: BM25 Score = 2.401393711092667
Document 1: BM25 Score = 1.9934863593049919
Document 9: BM25 Score = 1.372439303079609
Document 11: BM25 Score = 1.3150698709127657
Document 2: BM25 Score = 1.215258527344929
Document 10: BM25 Score = 1.0589292702677755
Document 7: BM25 Score = 0.7545663662387143
Document 6: BM25 Score = 0.7403616808178212
Document 4: BM25 Score = 0.726726618107892
Document 3: BM25 Score = 0.0
Document 5: BM25 Score = 0.0


### 5. Implement the evaluation methods: precision, recall, precision@k, recall@k, NDCG@k. 

In [49]:
# Labels de référence (gold standard)
gold_standard = {1: 1, 2: 1, 3: 0, 4: 1, 5: 0, 6: 1, 7: 1, 8: 1, 9: 1, 10: 0, 11: 0}

# Fonction pour calculer la précision globale
def precision(ranked_dict):
    retrieved_documents = [doc_id for doc_id, score in ranked_dict.items() if score != 0]
    relevant_and_retrieved = len(set(retrieved_documents) & set(gold_standard.keys()))
    total_retrieved_documents = len(retrieved_documents)
    return relevant_and_retrieved / total_retrieved_documents

# Fonction pour calculer le rappel global
def recall(ranked_dict):
    relevant_documents = [doc_id for doc_id, label in gold_standard.items() if label == 1]
    retrieved_documents = [doc_id for doc_id, score in ranked_dict.items() if score != 0]
    true_positives = len(set(retrieved_documents) & set(relevant_documents))
    total_relevant_documents = len(relevant_documents)
    return true_positives / total_relevant_documents

# Fonction pour calculer la précision à k
def precision_at_k(k, ranked_dict):
    retrieved_documents = [doc_id for doc_id, score in ranked_dict.items() if score != 0][:k]
    relevant_documents = [doc_id for doc_id, label in gold_standard.items() if label == 1]
    true_positives = len(set(retrieved_documents) & set(relevant_documents))
    return true_positives / k

# Fonction pour calculer le rappel à k
def recall_at_k(k, ranked_dict):
    retrieved_documents = [doc_id for doc_id, score in ranked_dict.items() if score != 0][:k]
    relevant_documents = [doc_id for doc_id, label in gold_standard.items() if label == 1]
    true_positives = len(set(retrieved_documents) & set(relevant_documents))
    total_relevant_documents = len(relevant_documents)
    return true_positives / total_relevant_documents

# Fonction pour calculer le NDCG
def ndcg_at_k(k, ranked_dict):
    relevant_documents = [doc_id for doc_id, label in gold_standard.items() if label == 1]
    top_k_documents = [doc_id for doc_id, score in sorted(ranked_dict.items(), key=lambda x: x[1], reverse=True)[:k]]
    
    dcg = 0
    for i, doc_id in enumerate(top_k_documents, 1):
        relevance = 1 if gold_standard[doc_id] == 1 else 0
        dcg += (2 ** relevance - 1) / (math.log2(i + 1))
    
    idcg = 0
    for i in range(1, min(k, len(relevant_documents)) + 1):
        idcg += 1 / math.log2(i + 1)
    
    return dcg / idcg


In [50]:
# POUR IF-IDF

# Convertir les scores en un dictionnaire
ranked_dict_tfidf = {int(doc_id): score for doc_id, score in ranked_documents_tfidf}

# Calcul des mesures d'évaluation pour k=5
k = 5
precision_tf = precision(ranked_dict_tfidf)
recall_tf = recall(ranked_dict_tfidf)
precision_5_tf = precision_at_k(k, ranked_dict_tfidf)
recall_5_tf = recall_at_k(k, ranked_dict_tfidf)
ndcg_5_tf = ndcg_at_k(k, ranked_dict_tfidf)

print(f"Precision: {precision_tf}")
print(f"Recall: {recall_tf}")
print(f"Precision@{k}: {precision_5_tf}")
print(f"Recall@{k}: {recall_5_tf}")
print(f"NDCG@{k}: {ndcg_5_tf}")


Precision: 1.0
Recall: 1.0
Precision@5: 0.8
Recall@5: 0.5714285714285714
NDCG@5: 0.8539316501572937


In [51]:
# POUR BM25
# Scores BM25 classés
ranked_documents_bm25 = [
    ('8', 2.401393711092667),
    ('1', 1.9934863593049919),
    ('9', 1.372439303079609),
    ('11', 1.3150698709127657),
    ('2', 1.215258527344929),
    ('10', 1.0589292702677755),
    ('7', 0.7545663662387143),
    ('6', 0.7403616808178212),
    ('4', 0.726726618107892),
    ('3', 0.0),
    ('5', 0.0)
]

# Conversion en un dictionnaire avec les mêmes clés que TF-IDF
ranked_dict_bm25 = {int(doc_id): score for doc_id, score in ranked_documents_bm25}

# Calcul des mesures d'évaluation pour k=5
k = 5
precision_bm = precision(ranked_dict_bm25)
recall_bm = recall(ranked_dict_bm25)
precision_5_bm = precision_at_k(k, ranked_dict_bm25)
recall_5_bm = recall_at_k(k, ranked_dict_bm25)
ndcg_5_bm = ndcg_at_k(k, ranked_dict_bm25)

print(f"Precision: {precision_bm}")
print(f"Recall: {recall_bm}")
print(f"Precision@{k}: {precision_5_bm}")
print(f"Recall@{k}: {recall_5_bm}")
print(f"NDCG@{k}: {ndcg_5_bm}")


Precision: 1.0
Recall: 1.0
Precision@5: 0.8
Recall@5: 0.5714285714285714
NDCG@5: 0.8539316501572937


Remarque : dans les calculs de precision, recall, etc, on a les mêmes résultats pour tf-idf et bm25 car on a considéré un document comme étant pertinent si son score est différent de 0. Or tf-idf et bm-25 ont tout les deux 2 documents dont le score vaut 0. Ainsi, c'est normal qu'on a les mêmes résultats. Si à la place, on dit qu'un document est pertinent pour un certain seuil de score(par exemple score > 1), on va obtenir des résultats différents, tout simplement parce que tf-idf a 7 documents dont le score est supérieur à 1 alors que bm25 en a seulement 6.

### 6. Test your retrieval systems, using the data, query and gold standard provided. For the BM25 use k1=0, b=1, and k1=1,b1=0.5. For the two implementations (tf-idf, and BM25) compute the NDCG, precision and recall @2, @3, @4 and @5 and design the Precision-Recall curve. 

### 7. Report the results that you obtain from the evaluation of the different algorithms for the different metrics. Compare the algorithms, and drive your conclusions, and ideas for improvement of the algorithms and/or the test data set